In [1]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from time import sleep
import numpy as np
import jax
import jax.numpy as jnp
from src.qwen_jax import (QwenForCausalLM,
    create_qwen_config_from_pytorch
)


%load_ext autoreload
%autoreload 2

os.environ["CUDA_VISIBLE_DEVICES"] = ""
# Model Configuration - Qwen 2.5 0.5B
MODEL_NAME = "Qwen/Qwen2.5-7B"  # Smaller 0.5B model for easier loading

# Device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Target Device: {device}")

# Model loading configuration (0.5B model is much smaller)
model_config = {
    "device_map": "auto",
    "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
    "trust_remote_code": False,
}

print(f"Loading: {MODEL_NAME}")

Target Device: cpu
Loading: Qwen/Qwen2.5-7B


In [2]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
print("Tokenizer loaded!")

# Set padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Loading tokenizer...
Tokenizer loaded!


In [3]:
# Load Qwen 2.5 0.5B Model
print("Loading model...")

try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_config)
    model.eval()
    print("Model loaded!")

    # Basic info
    param_count = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {param_count:,}")

except Exception as e:
    print(f"Error: {e}")
    model = None

`torch_dtype` is deprecated! Use `dtype` instead!


Loading model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded!
Parameters: 7,615,616,512


In [5]:
from transformers.models.qwen2.modeling_qwen2 import Qwen2ForCausalLM

In [6]:
def generate_text(prompt, max_new_tokens=10):
    inputs = tokenizer.encode(prompt, return_tensors="pt")
    inputs = inputs.to(next(model.parameters()).device)

    print(f"Prompt: '{prompt}'")
    print(f"Input Tokens: {inputs}")
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=10,
            temperature=.7,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    print(f"Output Tokens: {outputs} shape {outputs.shape}")
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Generated: '{generated_text}'")
    return generated_text


# Test the function
generate_text("How can I transform a pytorch model into jax?")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Prompt: 'How can I transform a pytorch model into jax?'
Input Tokens: tensor([[ 4340,   646,   358,  5165,   264,  4510, 27414,  1614,  1119,   502,
           706,    30]], device='cuda:0')
Output Tokens: tensor([[ 4340,   646,   358,  5165,   264,  4510, 27414,  1614,  1119,   502,
           706,    30,   358,   614,   264,  4510, 27414,  1614,   429,   358,
          1366,   311]], device='cuda:0') shape torch.Size([1, 22])
Generated: 'How can I transform a pytorch model into jax? I have a pytorch model that I want to'


'How can I transform a pytorch model into jax? I have a pytorch model that I want to'

In [6]:
# Additional Test
if model is not None and tokenizer is not None:
    print("Running one more test...")
    text = generate_text(
        "Python is a programming language that", max_new_tokens=40)
else:
    print("Model not loaded")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Running one more test...
Prompt: 'Python is a programming language that'
Input Tokens: tensor([[30280,   374,   264, 15473,  4128,   429]], device='cuda:0')
Output Tokens: tensor([[30280,   374,   264, 15473,  4128,   429,   374, 13570,  1483,   369,
          3482,  4401,    11,   821,  6358,    11]], device='cuda:0') shape torch.Size([1, 16])
Generated: 'Python is a programming language that is widely used for web development, data analysis,'


In [7]:
# print model architecture
print("Model Architecture:")
print(model)

Model Architecture:
Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_e

In [4]:
# print model architecture
print("Model Architecture:")
print(model)

Model Architecture:
Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-0

In [4]:
# Model Info
if model is not None:
    config = model.config
    print(f"Model Info:")
    print(f"   Architecture: {config.architectures}")
    print(f"   Hidden Size: {config.hidden_size}")
    print(f"   Layers: {config.num_hidden_layers}")
    print(f"   Attention Heads: {config.num_attention_heads}")
    print(f"   Vocab Size: {config.vocab_size:,}")
else:
    print("Model not loaded")

Model Info:
   Architecture: ['Qwen2ForCausalLM']
   Hidden Size: 3584
   Layers: 28
   Attention Heads: 28
   Vocab Size: 152,064


In [8]:
# Model Info
if model is not None:
    config = model.config
    print(f"Model Info:")
    print(f"   Architecture: {config.architectures}")
    print(f"   Hidden Size: {config.hidden_size}")
    print(f"   Layers: {config.num_hidden_layers}")
    print(f"   Attention Heads: {config.num_attention_heads}")
    print(f"   Vocab Size: {config.vocab_size:,}")
else:
    print("Model not loaded")

Model Info:
   Architecture: ['Qwen2ForCausalLM']
   Hidden Size: 896
   Layers: 24
   Attention Heads: 14
   Vocab Size: 151,936


In [9]:
config

Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 24,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "ro

## rotary empedding verification

In [10]:
from transformers.models.qwen2.modeling_qwen2 import Qwen2RotaryEmbedding

rot_embedding_torch = Qwen2RotaryEmbedding(
    config
)
temp_x = torch.randn(1, 10, config.hidden_size // config.num_attention_heads)
print(f"Temp X shape: {temp_x.shape}")
position_ids = torch.arange(temp_x.shape[1])[None, :]

for n, p in rot_embedding_torch.named_parameters():
    print(f"Parameter: {n} - Shape: {p.shape}")

cos, sin = rot_embedding_torch(temp_x, position_ids)
cos.shape

Temp X shape: torch.Size([1, 10, 64])


torch.Size([1, 10, 64])

In [11]:
from src.qwen_jax import RotaryEmbedding
from jax import numpy as jnp
import jax

key = jax.random.PRNGKey(0)

rot_embedding_jax = RotaryEmbedding(
    rope_theta=config.rope_theta,
    head_dim=config.hidden_size // config.num_attention_heads)

jax_x = jnp.array(temp_x.cpu().numpy())
jax_position_ids = jnp.array(position_ids.cpu().numpy())
_ = rot_embedding_jax.init(key, jax_x, jax_position_ids)
jax_cos, jax_sin = rot_embedding_jax.apply({}, jax_x, jax_position_ids)

print(f"JAX Cosine: {jax_cos.shape}, JAX Sine: {jax_sin.shape}")
# check if cos sin are equal
print(f"Cosine Equal: {jnp.allclose(cos.cpu().numpy(), jax_cos)}")
print(f"Sine Equal: {jnp.allclose(sin.cpu().numpy(), jax_sin)}")

JAX Cosine: (1, 10, 64), JAX Sine: (1, 10, 64)
Cosine Equal: True
Sine Equal: True


## Jax model definition

In [2]:
import jax
jax.devices()

E0829 08:38:55.859490 1761573 cuda_dnn.cc:523] Loaded runtime CuDNN library: 9.5.1 but source was compiled with: 9.8.0.  CuDNN library needs to have matching major version and equal or higher minor version. If using a binary install, upgrade your CuDNN library.  If building from sources, make sure the library loaded at runtime is compatible with the version specified during compile configuration.
E0829 08:38:55.864906 1761573 cuda_dnn.cc:523] Loaded runtime CuDNN library: 9.5.1 but source was compiled with: 9.8.0.  CuDNN library needs to have matching major version and equal or higher minor version. If using a binary install, upgrade your CuDNN library.  If building from sources, make sure the library loaded at runtime is compatible with the version specified during compile configuration.


[CudaDevice(id=0)]

In [12]:
# JAX/Flax Model Configuration Setup - Now using external module
import jax
import jax.numpy as jnp
from flax import linen as nn
import sys

from src.qwen_jax import (
    RMSNorm, RotaryEmbedding, apply_rotary_emb, rotate_half, repeat_kv,
    QwenMLP, QwenAttention, QwenDecoderLayer, QwenModel, QwenForCausalLM,
    create_qwen_config_from_pytorch
)

# Get config from PyTorch model for reference
config = model.config

print(f"Model Configuration:")
print(f"   Hidden Size: {config.hidden_size}")
print(f"   Vocab Size: {config.vocab_size}")
print(f"   Num Layers: {config.num_hidden_layers}")
print(f"   Num Heads: {config.num_attention_heads}")
print(f"   Num KV Heads: {config.num_key_value_heads}")
print(f"   Intermediate Size: {config.intermediate_size}")
print(f"   Max Position: {config.max_position_embeddings}")
print(f"   RMS Norm Eps: {config.rms_norm_eps}")
print(f"   Hidden Act: {config.hidden_act}")

# Store config values for JAX model
HIDDEN_SIZE = config.hidden_size  # 896
VOCAB_SIZE = config.vocab_size    # 151936
NUM_LAYERS = config.num_hidden_layers  # 24
NUM_HEADS = config.num_attention_heads  # 14
NUM_KV_HEADS = config.num_key_value_heads  # 2
INTERMEDIATE_SIZE = config.intermediate_size  # 4864
MAX_POSITION_EMBEDDINGS = config.max_position_embeddings  # 32768
RMS_NORM_EPS = config.rms_norm_eps  # 1e-06
HIDDEN_ACT = config.hidden_act  # "silu"
HEAD_DIM = HIDDEN_SIZE // NUM_HEADS  # 64

print(f"\nDerived values:")
print(f"   Head Dim: {HEAD_DIM}")
print(f"   Num KV Groups: {NUM_HEADS // NUM_KV_HEADS}")

print("\nJAX model components imported from qwen_jax.py!")
print("Available classes: RMSNorm, RotaryEmbedding, QwenMLP, QwenAttention, QwenDecoderLayer, QwenModel, QwenForCausalLM")
print("Available functions: apply_rotary_emb, rotate_half, repeat_kv, create_qwen_config_from_pytorch")

Model Configuration:
   Hidden Size: 896
   Vocab Size: 151936
   Num Layers: 24
   Num Heads: 14
   Num KV Heads: 2
   Intermediate Size: 4864
   Max Position: 32768
   RMS Norm Eps: 1e-06
   Hidden Act: silu

Derived values:
   Head Dim: 64
   Num KV Groups: 7

JAX model components imported from qwen_jax.py!
Available classes: RMSNorm, RotaryEmbedding, QwenMLP, QwenAttention, QwenDecoderLayer, QwenModel, QwenForCausalLM
Available functions: apply_rotary_emb, rotate_half, repeat_kv, create_qwen_config_from_pytorch


In [13]:
# Test individual JAX components (now imported from qwen_jax.py)
print("Testing imported JAX components...")

# Create test inputs for component testing
batch_size, seq_len = 2, 10
test_hidden_states = jnp.ones((batch_size, seq_len, HIDDEN_SIZE))

# Test RMSNorm
print("\nTesting RMSNorm...")
rms_norm = RMSNorm(eps=RMS_NORM_EPS)
rng = jax.random.PRNGKey(42)
rms_params = rms_norm.init(rng, test_hidden_states)
rms_output = rms_norm.apply(rms_params, test_hidden_states)
print(
    f"RMSNorm: input {test_hidden_states.shape} -> output {rms_output.shape}")

# Test RotaryEmbedding
print("\nTesting RotaryEmbedding...")
rope = RotaryEmbedding(head_dim=HEAD_DIM)
position_ids = jnp.arange(seq_len)[None, :].repeat(batch_size, axis=0)
rope_params = rope.init(rng, test_hidden_states, position_ids)
cos, sin = rope.apply(rope_params, test_hidden_states, position_ids)
print(f"RoPE: cos shape {cos.shape}, sin shape {sin.shape}")

# Test apply_rotary_emb function
print("\nTesting apply_rotary_emb...")
test_q = jnp.ones((batch_size, NUM_HEADS, seq_len, HEAD_DIM))
rotated_q = apply_rotary_emb(test_q, cos, sin)
print(f"Rotary emb: input {test_q.shape} -> output {rotated_q.shape}")

# Test repeat_kv function
print("\nTesting repeat_kv...")
test_kv = jnp.ones((batch_size, NUM_KV_HEADS, seq_len, HEAD_DIM))
repeated_kv = repeat_kv(test_kv, NUM_HEADS // NUM_KV_HEADS)
print(f"Repeat KV: input {test_kv.shape} -> output {repeated_kv.shape}")

# Test MLP
print("\nTesting QwenMLP...")
mlp = QwenMLP(hidden_size=HIDDEN_SIZE, intermediate_size=INTERMEDIATE_SIZE)
mlp_params = mlp.init(rng, test_hidden_states)
mlp_output = mlp.apply(mlp_params, test_hidden_states)
print(f"MLP: input {test_hidden_states.shape} -> output {mlp_output.shape}")

print("\n✅ All component tests passed!")
print("Components are ready for equivalence testing with PyTorch.")

Testing imported JAX components...

Testing RMSNorm...


RMSNorm: input (2, 10, 896) -> output (2, 10, 896)

Testing RotaryEmbedding...
RoPE: cos shape (2, 10, 64), sin shape (2, 10, 64)

Testing apply_rotary_emb...
Rotary emb: input (2, 14, 10, 64) -> output (2, 14, 10, 64)

Testing repeat_kv...
Repeat KV: input (2, 2, 10, 64) -> output (2, 14, 10, 64)

Testing QwenMLP...
MLP: input (2, 10, 896) -> output (2, 10, 896)

✅ All component tests passed!
Components are ready for equivalence testing with PyTorch.


In [14]:
# Test Full JAX Model (using imported classes)
print("Testing full JAX Qwen model...")

# Create model configuration from PyTorch config
jax_config = create_qwen_config_from_pytorch(config)
print("JAX model configuration:")
for key, value in jax_config.items():
    print(f"   {key}: {value}")

# Initialize full JAX model
print("\nInitializing QwenForCausalLM...")
jax_model = QwenForCausalLM(**jax_config)

# Create dummy input for testing
dummy_input_ids = jnp.ones((1, 10), dtype=jnp.int32)
rng = jax.random.PRNGKey(42)

# Initialize model parameters
print("Initializing model parameters...")
jax_params = jax_model.init(rng, dummy_input_ids)

# Count parameters
n_params_jax = sum(p.size for p in jax.tree.leaves(jax_params))
print(f"JAX model parameters: {n_params_jax:,}")

# Test forward pass
print("Testing forward pass...")
logits = jax_model.apply(jax_params, dummy_input_ids, train=False)
print(f"Input shape: {dummy_input_ids.shape}")
print(f"Output shape: {logits.shape}")
print(f"Expected shape: (1, 10, {VOCAB_SIZE})")

print("\n✅ Full JAX model test passed!")
print("Model is ready for equivalence testing and weight conversion.")

Testing full JAX Qwen model...
JAX model configuration:
   vocab_size: 151936
   hidden_size: 896
   num_hidden_layers: 24
   num_attention_heads: 14
   num_key_value_heads: 2
   intermediate_size: 4864
   hidden_act: silu
   max_position_embeddings: 32768
   rms_norm_eps: 1e-06
   attention_dropout: 0.0
   tie_word_embeddings: True
   rope_theta: 1000000.0

Initializing QwenForCausalLM...
Initializing model parameters...
JAX model parameters: 494,032,768
Testing forward pass...
Input shape: (1, 10)
Output shape: (1, 10, 151936)
Expected shape: (1, 10, 151936)

✅ Full JAX model test passed!
Model is ready for equivalence testing and weight conversion.


In [15]:

# Basic info
param_count = sum(p.numel() for p in model.parameters())

jax_param_count = sum(p.size for p in jax.tree.leaves(jax_params))
print(f"PyTorch Model Parameters: {param_count:,}")
print(f"JAX Model Parameters: {jax_param_count:,}")

PyTorch Model Parameters: 494,032,768
JAX Model Parameters: 494,032,768


In [16]:
for n, p in model.named_parameters():
    print(n, p.shape)

model.embed_tokens.weight torch.Size([151936, 896])
model.layers.0.self_attn.q_proj.weight torch.Size([896, 896])
model.layers.0.self_attn.q_proj.bias torch.Size([896])
model.layers.0.self_attn.k_proj.weight torch.Size([128, 896])
model.layers.0.self_attn.k_proj.bias torch.Size([128])
model.layers.0.self_attn.v_proj.weight torch.Size([128, 896])
model.layers.0.self_attn.v_proj.bias torch.Size([128])
model.layers.0.self_attn.o_proj.weight torch.Size([896, 896])
model.layers.0.mlp.gate_proj.weight torch.Size([4864, 896])
model.layers.0.mlp.up_proj.weight torch.Size([4864, 896])
model.layers.0.mlp.down_proj.weight torch.Size([896, 4864])
model.layers.0.input_layernorm.weight torch.Size([896])
model.layers.0.post_attention_layernorm.weight torch.Size([896])
model.layers.1.self_attn.q_proj.weight torch.Size([896, 896])
model.layers.1.self_attn.q_proj.bias torch.Size([896])
model.layers.1.self_attn.k_proj.weight torch.Size([128, 896])
model.layers.1.self_attn.k_proj.bias torch.Size([128])
mo

In [17]:
for n, p in jax.tree.flatten_with_path(jax_params)[0]:
    name = "/".join((nn.key for nn in n))
    print(f"JAX Param: {name} - Size: {p.size}, Shape: {p.shape}")

JAX Param: params/embed_tokens/embedding - Size: 136134656, Shape: (151936, 896)
JAX Param: params/layers_0/input_layernorm/weight - Size: 896, Shape: (896,)
JAX Param: params/layers_0/mlp/down_proj/kernel - Size: 4358144, Shape: (4864, 896)
JAX Param: params/layers_0/mlp/gate_proj/kernel - Size: 4358144, Shape: (896, 4864)
JAX Param: params/layers_0/mlp/up_proj/kernel - Size: 4358144, Shape: (896, 4864)
JAX Param: params/layers_0/post_attention_layernorm/weight - Size: 896, Shape: (896,)
JAX Param: params/layers_0/self_attn/k_proj/bias - Size: 128, Shape: (128,)
JAX Param: params/layers_0/self_attn/k_proj/kernel - Size: 114688, Shape: (896, 128)
JAX Param: params/layers_0/self_attn/o_proj/kernel - Size: 802816, Shape: (896, 896)
JAX Param: params/layers_0/self_attn/q_proj/bias - Size: 896, Shape: (896,)
JAX Param: params/layers_0/self_attn/q_proj/kernel - Size: 802816, Shape: (896, 896)
JAX Param: params/layers_0/self_attn/v_proj/bias - Size: 128, Shape: (128,)
JAX Param: params/layer

## Parameter Transfer from PyTorch to JAX

Now we'll implement a robust parameter mapping system to transfer weights from the PyTorch model to the JAX model structure, then compare forward passes.

In [22]:
import transformers
n_params = 0
transpose_mask = []
for m in model.modules():
    module_n_params = len(list(m.parameters(recurse=False)))
    if module_n_params > 0:
        print(f"\t{type(m)}, Num direct params: {module_n_params}")
    else:
        print(f"{type(m)}")
    n_params += module_n_params
print(f"Total modules: {n_params}")

<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
<class 'transformers.models.qwen2.modeling_qwen2.Qwen2Model'>
	<class 'torch.nn.modules.sparse.Embedding'>, Num direct params: 1
<class 'torch.nn.modules.container.ModuleList'>
<class 'transformers.models.qwen2.modeling_qwen2.Qwen2DecoderLayer'>
<class 'transformers.models.qwen2.modeling_qwen2.Qwen2Attention'>
	<class 'torch.nn.modules.linear.Linear'>, Num direct params: 2
	<class 'torch.nn.modules.linear.Linear'>, Num direct params: 2
	<class 'torch.nn.modules.linear.Linear'>, Num direct params: 2
	<class 'torch.nn.modules.linear.Linear'>, Num direct params: 1
<class 'transformers.models.qwen2.modeling_qwen2.Qwen2MLP'>
	<class 'torch.nn.modules.linear.Linear'>, Num direct params: 1
	<class 'torch.nn.modules.linear.Linear'>, Num direct params: 1
	<class 'torch.nn.modules.linear.Linear'>, Num direct params: 1
<class 'torch.nn.modules.activation.SiLU'>
	<class 'transformers.models.qwen2.modeling_qwen2.Qwen2RMSNorm'>, Num

In [23]:
from transformers.models.qwen2.modeling_qwen2 import Qwen2RMSNorm

In [24]:
for n, p in model.named_parameters():
    print(n, p.shape)

model.embed_tokens.weight torch.Size([151936, 896])
model.layers.0.self_attn.q_proj.weight torch.Size([896, 896])
model.layers.0.self_attn.q_proj.bias torch.Size([896])
model.layers.0.self_attn.k_proj.weight torch.Size([128, 896])
model.layers.0.self_attn.k_proj.bias torch.Size([128])
model.layers.0.self_attn.v_proj.weight torch.Size([128, 896])
model.layers.0.self_attn.v_proj.bias torch.Size([128])
model.layers.0.self_attn.o_proj.weight torch.Size([896, 896])
model.layers.0.mlp.gate_proj.weight torch.Size([4864, 896])
model.layers.0.mlp.up_proj.weight torch.Size([4864, 896])
model.layers.0.mlp.down_proj.weight torch.Size([896, 4864])
model.layers.0.input_layernorm.weight torch.Size([896])
model.layers.0.post_attention_layernorm.weight torch.Size([896])
model.layers.1.self_attn.q_proj.weight torch.Size([896, 896])
model.layers.1.self_attn.q_proj.bias torch.Size([896])
model.layers.1.self_attn.k_proj.weight torch.Size([128, 896])
model.layers.1.self_attn.k_proj.bias torch.Size([128])
mo

In [ ]:
from flax.core import freeze, unfreeze
import numpy as np

def create_parameter_mapping():
    """
    Create a mapping dictionary that defines how JAX pytree parameters 
    correspond to PyTorch parameters.
    
    Returns:
        tuple: With: (Mapping from JAX parameter paths to PyTorch parameter names, and transpose info)
    """
    mapping = {}
    
    # Embedding layer
    mapping["params/embed_tokens/embedding"] = (f"model.embed_tokens.weight", (0,1))
    
    # Transformer layers
    for layer_idx in range(NUM_LAYERS):
        prefix_torch = f"model.layers.{layer_idx}"
        prefix_jax = f"params/layers_{layer_idx}"
        
        # Attention weights
        mapping[f"{prefix_jax}/self_attn/q_proj/kernel"] = (f"{prefix_torch}.self_attn.q_proj.weight", (1,0))
        mapping[f"{prefix_jax}/self_attn/q_proj/bias"] = (f"{prefix_torch}.self_attn.q_proj.bias", (0,))
        mapping[f"{prefix_jax}/self_attn/k_proj/kernel"] = (f"{prefix_torch}.self_attn.k_proj.weight", (1,0))
        mapping[f"{prefix_jax}/self_attn/k_proj/bias"] = (f"{prefix_torch}.self_attn.k_proj.bias", (0,))
        mapping[f"{prefix_jax}/self_attn/v_proj/kernel"] = (f"{prefix_torch}.self_attn.v_proj.weight", (1,0))
        mapping[f"{prefix_jax}/self_attn/v_proj/bias"] = (f"{prefix_torch}.self_attn.v_proj.bias", (0,))
        mapping[f"{prefix_jax}/self_attn/o_proj/kernel"] = (f"{prefix_torch}.self_attn.o_proj.weight", (1,0))
        
        # Layer norms
        mapping[f"{prefix_jax}/input_layernorm/weight"] = (f"{prefix_torch}.input_layernorm.weight", (0,))
        mapping[f"{prefix_jax}/post_attention_layernorm/weight"] = (f"{prefix_torch}.post_attention_layernorm.weight", (0,))

        # MLP weights
        mapping[f"{prefix_jax}/mlp/gate_proj/kernel"] = (f"{prefix_torch}.mlp.gate_proj.weight", (1,0))
        mapping[f"{prefix_jax}/mlp/up_proj/kernel"] = (f"{prefix_torch}.mlp.up_proj.weight", (1,0))
        mapping[f"{prefix_jax}/mlp/down_proj/kernel"] = (f"{prefix_torch}.mlp.down_proj.weight", (1,0))

    # Final layer norm
    mapping["params/norm/weight"] = (f"model.norm.weight", (0,))

    # Language model head (if using tied embeddings, this might not exist)
    if hasattr(model, 'lm_head') and model.lm_head is not None:
        mapping["params/lm_head/kernel"] = (f"lm_head.weight", (0,1))

    return mapping

def get_dtype_from_tensor(torch_tensor):
    if torch_tensor.dtype == torch.float32:
        return jnp.float32
    elif torch_tensor.dtype == torch.float16:
        return jnp.float16
    elif torch_tensor.dtype == torch.bfloat16:
        return jnp.bfloat16
    else:
        raise ValueError(f"Unsupported torch dtype: {torch_tensor.dtype}")

# Create the reversed mapping (JAX -> PyTorch)
param_mapping = create_parameter_mapping()
print(f"Created parameter mapping with {len(param_mapping)} entries")
print("\nSample mappings (JAX -> PyTorch):")
for i, (jax_path, pytorch_name) in enumerate(list(param_mapping.items())[:5]):
    print(f"  {jax_path} -> {pytorch_name}")
print("  ...")

Created parameter mapping with 291 entries

Sample mappings (JAX -> PyTorch):
  params/embed_tokens/embedding -> ('model.embed_tokens.weight', (0, 1))
  params/layers_0/self_attn/q_proj/kernel -> ('model.layers.0.self_attn.q_proj.weight', (1, 0))
  params/layers_0/self_attn/q_proj/bias -> ('model.layers.0.self_attn.q_proj.bias', (0,))
  params/layers_0/self_attn/k_proj/kernel -> ('model.layers.0.self_attn.k_proj.weight', (1, 0))
  params/layers_0/self_attn/k_proj/bias -> ('model.layers.0.self_attn.k_proj.bias', (0,))
  ...


In [26]:
pytorch_params = dict(model.named_parameters())
jax_params = unfreeze(jax_params)

tree_flat, tree_struct = jax.tree.flatten_with_path(jax_params)

new_jax_params = []
for (n, p) in tree_flat:
    jax_path = "/".join((nn.key for nn in n))
    pytorch_name, transpose_order = param_mapping.get(jax_path, (None,None))
    
    if pytorch_name is None:
        print(f"WARNING: No mapping found for JAX path '{jax_path}'")
        continue
    
    pp = pytorch_params.get(pytorch_name, None)
    if pp is None:
        print(f"WARNING: No PyTorch parameter found for '{pytorch_name}'")
        continue

    dtype = get_dtype_from_tensor(pp)
    try:
        pp = pp.detach().cpu().numpy().transpose(transpose_order)
    except Exception as e:
        print(f"WARNING: Error processing '{pytorch_name}': {e}")
        print(f"\tTranspose order: {transpose_order}, Original shape: {pp.shape}")
        continue
    if p.shape != pp.shape:
        print(f"WARNING: Shape mismatch for {pytorch_name}: JAX {p.shape} vs PyTorch {pp.shape}")
        continue

    new_jax_params.append(jnp.array(pp, dtype=dtype))

jax_params = jax.tree.unflatten(tree_struct, new_jax_params)
del new_jax_params



### compare embeddings 
(shared accros last and first layer)

In [113]:
model.model.embed_tokens.weight.shape, model.lm_head.weight.shape

(torch.Size([151936, 896]), torch.Size([151936, 896]))

In [114]:
torch.allclose(model.lm_head.weight, model.model.embed_tokens.weight)

True

In [115]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbe

In [117]:
import gc
gc.collect()  # Clear memory before comparison

730

In [192]:
test_input_ids = np.array([[1,2,3,4,5,6,7,8,9,10]])
pytorch_input = torch.from_numpy(np.array(test_input_ids)).to(next(model.parameters()).device)
torch_embed_tokens = model.model.embed_tokens(pytorch_input)
torch_embed_tokens

tensor([[[-1.4648e-02,  6.0320e-05, -1.8433e-02,  ..., -1.6937e-03,
           3.1128e-03, -8.4229e-03],
         [-3.6621e-02, -9.8267e-03,  6.9885e-03,  ..., -7.7515e-03,
          -1.7090e-02, -1.0490e-04],
         [-9.3994e-03, -1.0864e-02, -1.4893e-02,  ..., -1.6937e-03,
           1.4832e-02, -1.6602e-02],
         ...,
         [-2.1606e-02,  1.3367e-02, -9.5825e-03,  ...,  2.2583e-02,
           1.8921e-03, -3.7537e-03],
         [-5.4626e-03, -1.1841e-02, -9.1553e-03,  ...,  1.8311e-03,
           7.4158e-03, -1.7090e-02],
         [-3.2959e-02,  3.1891e-03, -7.5073e-03,  ...,  1.9775e-02,
          -4.1504e-03, -4.1504e-03]]], device='cuda:0', dtype=torch.float16,
       grad_fn=<EmbeddingBackward0>)

In [193]:
model_def = jax_model.tabulate(jax.random.PRNGKey(0), test_input_ids, compute_flops=False, compute_vjp_flops=False, console_kwargs={"width": 200})
print(model_def)


                                                        QwenForCausalLM Summary                                                        
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ path                               ┃ module           ┃ inputs              ┃ outputs              ┃ params                         ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│                                    │ QwenForCausalLM  │ int32[1,10]         │ float32[1,10,151936] │                                │
├────────────────────────────────────┼──────────────────┼─────────────────────┼──────────────────────┼────────────────────────────────┤
│ embed_tokens                       │ Embed            │ int32[1,10]         │ float32[1,10,896]    │ embedding: float32[151936,896] │
│                                    │         

In [194]:
# Execute only the JAX embed_tokens embedding layer
embed_tokens = jax_params['params']['embed_tokens']
input_ids = test_input_ids
filter_fn = lambda mdl, md_name: isinstance(mdl, nn.Embed) and md_name == "__call__"

out, inter = jax_model.apply(jax_params, input_ids, train=False, capture_intermediates=filter_fn, mutable=['intermediates',])
print(out.shape)
embed_tokens_inter_jax = inter['intermediates']['embed_tokens']['__call__'][0]
print(f"Jax embed_tokens output shape: {embed_tokens_inter_jax.shape}; should match torch: {torch_embed_tokens.shape}")
print(f"Jax dtype: {embed_tokens_inter_jax.dtype}, Torch dtype: {torch_embed_tokens.dtype}")

# Compare JAX and PyTorch embed_tokens outputs
jax_embed = embed_tokens_inter_jax
torch_embed = torch_embed_tokens.detach().cpu().numpy()

if jax_embed.shape == torch_embed.shape:
    diff = np.abs(jax_embed - torch_embed)
    print(f"Embed comparison: max diff {diff.max():.6f}, mean diff {diff.mean():.6f}")
else:
    print(f"Shape mismatch: JAX {jax_embed.shape} vs Torch {torch_embed.shape}")


(1, 10, 151936)
Jax embed_tokens output shape: (1, 10, 896); should match torch: torch.Size([1, 10, 896])
Jax dtype: float16, Torch dtype: torch.float16
Embed comparison: max diff 0.000000, mean diff 0.000000


### compare different layers

In [195]:

jax_model = QwenForCausalLM(**jax_config)

In [196]:
# Execute only the JAX embed_tokens embedding layer
input_ids = test_input_ids
filter_fn = lambda mdl, md_name: md_name == "__call__"

out, inter = jax_model.apply(jax_params, input_ids, train=False, capture_intermediates=filter_fn, mutable=['intermediates',])
print(out.shape)
inter_jax = inter['intermediates']['layers_0']['self_attn']['__call__'][0]
print(f"Jax embed_tokens output shape: {inter_jax.shape}, dtype: {inter_jax.dtype}")


(1, 10, 151936)
Jax embed_tokens output shape: (1, 10, 896), dtype: float16


In [197]:
inter['intermediates']['layers_0']['self_attn']['q_proj']['__call__'][0].dtype

dtype('float16')

In [198]:
inter['intermediates']['layers_0']['self_attn']['rotary_emb']['__call__'][0][0].dtype

dtype('float16')

In [ ]:
# inter['intermediates']['layers_0']['self_attn']['attn_weights_soft']

KeyError: 'attn_weights_soft'

In [201]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbe

In [213]:
from transformers.models.qwen2.modeling_qwen2 import Qwen2Attention

In [214]:
## compare Qwen2DecoderLayer

# pytorch
# Storage for activations
activations = {}

# Define a hook function
def forward_hook(module, input, output):
    print(f"Inside {module.__class__.__name__} forward")
    activations[module.__class__.__name__] = output.detach()
try:
    hook_handle = model.model.layers[0].register_forward_hook(forward_hook)

    pytorch_input = torch.from_numpy(np.array(test_input_ids)).to(next(model.parameters()).device)

    with torch.no_grad():
        model.eval()
        pytorch_logits = model(pytorch_input).logits
finally:
    hook_handle.remove()  # Clean up the hook
torch_atten = activations['Qwen2DecoderLayer'].detach().cpu().numpy()

# jax
out, inter = jax_model.apply(jax_params, test_input_ids, train=False, capture_intermediates=True, mutable=['intermediates',])
print(out.shape)
jax_atten = inter['intermediates']['layers_0']['__call__'][0]

# verify shapes and values
print(f"PyTorch attention shape: {torch_atten.shape}")
print(f"JAX attention shape: {jax_atten.shape}")
print(f"Shapes match: {torch_atten.shape == jax_atten.shape}")
print(f"Values close: {np.allclose(torch_atten, jax_atten, atol=1e-3)}")    
diff = np.array(inter['intermediates']['layers_0']['__call__'][0]) - activations['Qwen2DecoderLayer'].detach().cpu().numpy()
print(f"Max difference: {np.abs(diff).max()}, Mean difference: {np.abs(diff).mean()}, Std difference: {np.abs(diff).std()}")



Inside Qwen2DecoderLayer forward
(1, 10, 151936)
PyTorch attention shape: (1, 10, 896)
JAX attention shape: (1, 10, 896)
Shapes match: True
Values close: False
Max difference: 0.0390625, Mean difference: 0.0007848739624023438, Std difference: 0.0011453628540039062


In [253]:
## compare Qwen2Attention

# pytorch
# Storage for activations
activations = {}
layer_name = "input_layernorm"

# Define a hook function
def forward_hook(module, input, output):
    print(f"Inside {module.__class__.__name__} forward")
    activations[layer_name] = output[0]
    activations['input'] = input
try:
    hook_handle = model.model.layers[0].input_layernorm.register_forward_hook(forward_hook)

    pytorch_input = torch.from_numpy(np.array(test_input_ids)).to(next(model.parameters()).device)

    with torch.no_grad():
        model.eval()
        pytorch_logits = model(pytorch_input).logits
finally:
    print("Removing hook")
    hook_handle.remove()  # Clean up the hook
torch_atten = activations[layer_name].detach().cpu().numpy()

# jax
out, inter = jax_model.apply(jax_params, test_input_ids, train=False, capture_intermediates=True, mutable=['intermediates',])
print(out.shape)
jax_atten = inter['intermediates']['layers_0'][layer_name]['__call__'][0]

# verify shapes and values
print(f"PyTorch attention shape: {torch_atten.shape}")
print(f"JAX attention shape: {jax_atten.shape}")
print(f"Shapes match: {torch_atten.shape == jax_atten.shape}")
print(f"Values close: {np.allclose(torch_atten, jax_atten, rtol=1e-5)}")    
diff = np.array(jax_atten) - activations[layer_name].detach().cpu().numpy()
print(f"Max difference: {np.abs(diff).max()}, Mean difference: {np.abs(diff).mean()}, Std difference: {np.abs(diff).std()}")
activations


Inside Qwen2RMSNorm forward
Removing hook
(1, 10, 151936)
PyTorch attention shape: (10, 896)
JAX attention shape: (1, 10, 896)
Shapes match: False
Values close: True
Max difference: 0.0, Mean difference: 0.0, Std difference: 0.0


{'input_layernorm': tensor([[ 4.0771e-02,  1.5903e-04,  2.7161e-02,  ...,  4.9934e-03,
          -4.7455e-03, -8.3374e-02],
         [ 1.0992e-01, -2.7939e-02, -1.1101e-02,  ...,  2.4628e-02,
           2.8076e-02, -1.1196e-03],
         [ 2.7618e-02, -3.0243e-02,  2.3163e-02,  ...,  5.2681e-03,
          -2.3880e-02, -1.7346e-01],
         ...,
         [ 6.5918e-02,  3.8635e-02,  1.5472e-02,  ..., -7.2937e-02,
          -3.1624e-03, -4.0741e-02],
         [ 1.6205e-02, -3.3234e-02,  1.4366e-02,  ..., -5.7449e-03,
          -1.2039e-02, -1.8030e-01],
         [ 1.0114e-01,  9.2697e-03,  1.2192e-02,  ..., -6.4209e-02,
           6.9771e-03, -4.5288e-02]], device='cuda:0', dtype=torch.float16),
 'input': (tensor([[[-1.4648e-02,  6.0320e-05, -1.8433e-02,  ..., -1.6937e-03,
             3.1128e-03, -8.4229e-03],
           [-3.6621e-02, -9.8267e-03,  6.9885e-03,  ..., -7.7515e-03,
            -1.7090e-02, -1.0490e-04],
           [-9.3994e-03, -1.0864e-02, -1.4893e-02,  ..., -1.6937e-03,


In [302]:
## compare Qwen2Attention

# pytorch
# Storage for activations
activations = {}
layer_name = "self_attn"

# Define a hook function
def forward_hook(module, input, output):
    print(f"Inside {module.__class__.__name__} forward")
    activations[layer_name] = output[0]
    activations['input'] = input
try:
    hook_handle = model.model.layers[0].self_attn.register_forward_hook(forward_hook)

    pytorch_input = torch.from_numpy(np.array(test_input_ids)).to(next(model.parameters()).device)

    with torch.no_grad():
        model.eval()
        pytorch_logits = model(pytorch_input).logits
finally:
    print("Removing hook")
    hook_handle.remove()  # Clean up the hook
torch_atten = activations[layer_name].detach().cpu().numpy()

# jax
out, inter = jax_model.apply(jax_params, test_input_ids, train=False, capture_intermediates=True, mutable=['intermediates',])
print(out.shape)
jax_atten = inter['intermediates']['layers_0'][layer_name]['__call__'][0]

# verify shapes and values
print(f"PyTorch attention shape: {torch_atten.shape}")
print(f"JAX attention shape: {jax_atten.shape}")
print(f"Shapes match: {torch_atten.shape == jax_atten.shape}")
print(f"Values close: {np.allclose(torch_atten, np.array(jax_atten), atol=1e-3)}")    
diff = np.array(jax_atten) - activations[layer_name].detach().cpu().numpy()
print(f"Max difference: {np.abs(diff).max()}, Mean difference: {np.abs(diff).mean()}, Std difference: {np.abs(diff).std()}")



Inside Qwen2Attention forward
Removing hook
(1, 10, 151936)
PyTorch attention shape: (1, 10, 896)
JAX attention shape: (1, 10, 896)
Shapes match: True
Values close: True
Max difference: 0.0007171630859375, Mean difference: 6.777048110961914e-05, Std difference: 0.0


In [254]:
## compare SelfAttention

# pytorch
# Storage for activations
activations = {}
layer_name = "q_proj"

# Define a hook function
def forward_hook(module, input, output):
    print(f"Inside {module.__class__.__name__} forward")
    activations[layer_name] = output
    activations['input'] = input
try:
    hook_handle = model.model.layers[0].self_attn.q_proj.register_forward_hook(forward_hook)

    pytorch_input = torch.from_numpy(np.array(test_input_ids)).to(next(model.parameters()).device)

    with torch.no_grad():
        model.eval()
        pytorch_logits = model(pytorch_input).logits
finally:
    print("Removing hook")
    hook_handle.remove()  # Clean up the hook
torch_atten = activations[layer_name].detach().cpu().numpy()

# jax
out, inter = jax_model.apply(jax_params, test_input_ids, train=False, capture_intermediates=True, mutable=['intermediates',])
print(out.shape)
jax_atten = inter['intermediates']['layers_0']['self_attn'][layer_name]['__call__'][0]

# verify shapes and values
print(f"PyTorch attention shape: {torch_atten.shape}")
print(f"JAX attention shape: {jax_atten.shape}")
print(f"Shapes match: {torch_atten.shape == jax_atten.shape}")
print(f"Values close: {np.allclose(torch_atten, jax_atten, rtol=1e-5)}")    
diff = np.array(jax_atten) - activations[layer_name].detach().cpu().numpy()
print(f"Max difference: {np.abs(diff).max()}, Mean difference: {np.abs(diff).mean()}, Std difference: {np.abs(diff).std()}")
activations


Inside Linear forward
Removing hook
(1, 10, 151936)
PyTorch attention shape: (1, 10, 896)
JAX attention shape: (1, 10, 896)
Shapes match: True
Values close: False
Max difference: 0.015625, Mean difference: 0.0001227855682373047, Std difference: 0.0006461143493652344


{'q_proj': tensor([[[ 0.0899,  0.3264,  0.0979,  ...,  1.4453,  0.9580, -1.0498],
          [ 0.0220,  0.2264,  0.0080,  ...,  1.2158,  0.9390, -0.8726],
          [ 0.0579,  0.2386,  0.0594,  ...,  1.2383,  1.0273, -0.9570],
          ...,
          [ 0.0310,  0.2573,  0.0345,  ...,  1.5820,  0.9888, -0.7876],
          [ 0.0190,  0.2246, -0.0083,  ...,  1.0938,  0.8667, -0.9258],
          [ 0.0138,  0.1996, -0.0502,  ...,  1.1221,  0.9224, -0.9473]]],
        device='cuda:0', dtype=torch.float16),
 'input': (tensor([[[ 4.0771e-02,  1.5903e-04,  2.7161e-02,  ...,  4.9934e-03,
            -4.7455e-03, -8.3374e-02],
           [ 1.0992e-01, -2.7939e-02, -1.1101e-02,  ...,  2.4628e-02,
             2.8076e-02, -1.1196e-03],
           [ 2.7618e-02, -3.0243e-02,  2.3163e-02,  ...,  5.2681e-03,
            -2.3880e-02, -1.7346e-01],
           ...,
           [ 6.5918e-02,  3.8635e-02,  1.5472e-02,  ..., -7.2937e-02,
            -3.1624e-03, -4.0741e-02],
           [ 1.6205e-02, -3.3234e

In [243]:
inter['intermediates']['layers_0']

{'input_layernorm': {'__call__': (Array([[[ 4.0771e-02,  1.5903e-04,  2.7161e-02, ...,  4.9934e-03,
            -4.7455e-03, -8.3374e-02],
           [ 1.0992e-01, -2.7939e-02, -1.1101e-02, ...,  2.4628e-02,
             2.8076e-02, -1.1196e-03],
           [ 2.7618e-02, -3.0243e-02,  2.3163e-02, ...,  5.2681e-03,
            -2.3880e-02, -1.7346e-01],
           ...,
           [ 6.5918e-02,  3.8635e-02,  1.5472e-02, ..., -7.2937e-02,
            -3.1624e-03, -4.0741e-02],
           [ 1.6205e-02, -3.3234e-02,  1.4366e-02, ..., -5.7449e-03,
            -1.2039e-02, -1.8030e-01],
           [ 1.0114e-01,  9.2697e-03,  1.2192e-02, ..., -6.4209e-02,
             6.9771e-03, -4.5288e-02]]], dtype=float16),)},
 'self_attn': {'q_proj': {'__call__': (Array([[[ 0.0899  ,  0.3262  ,  0.0979  , ...,  1.445   ,  0.958   ,
             -1.05    ],
            [ 0.02197 ,  0.2264  ,  0.007996, ...,  1.216   ,  0.939   ,
             -0.8726  ],
            [ 0.05786 ,  0.2386  ,  0.05945 , ...,  1

In [293]:
t_fc = torch.nn.Linear(in_features=43, out_features=444, dtype=torch.float16)

kernel = t_fc.weight.detach().cpu().numpy()
bias = t_fc.bias.detach().cpu().numpy()

# [outC, inC] -> [inC, outC]
kernel = jnp.transpose(kernel, (1, 0))

key = jax.random.key(0)
x = jax.random.normal(key, (1, kernel.shape[0]), dtype=jnp.float16)

variables = {'params': {'kernel': kernel, 'bias': bias}}
j_fc = nn.Dense(features=kernel.shape[1])
j_out = j_fc.apply(variables, x)

t_x = torch.from_numpy(np.array(x))
t_out = t_fc(t_x)
t_out = t_out.detach().cpu().numpy()

print(f"abs mean diff: {np.abs(j_out - t_out).mean()}")
np.allclose(j_out, t_out, rtol=1e-5)

abs mean diff: 0.000125885009765625


False

In [236]:
## compare Qwen2Attention

# pytorch
# Storage for activations
activations = {}
layer_name = "self_attn"

# Define a hook function
def forward_hook(module, input, output):
    print(f"Inside {module.__class__.__name__} forward")
    attn_output, attn_weights = output
    activations[layer_name] = attn_output.detach()
    activations['input'] = input
try:
    hook_handle = model.model.layers[0].self_attn.register_forward_hook(forward_hook)

    pytorch_input = torch.from_numpy(np.array(test_input_ids)).to(next(model.parameters()).device)

    with torch.no_grad():
        model.eval()
        pytorch_logits = model(pytorch_input).logits
finally:
    print("Removing hook")
    hook_handle.remove()  # Clean up the hook
torch_atten = activations[layer_name].detach().cpu().numpy()

# jax
out, inter = jax_model.apply(jax_params, test_input_ids, train=False, capture_intermediates=True, mutable=['intermediates',])
print(out.shape)
jax_atten = inter['intermediates']['layers_0'][layer_name]['__call__'][0]

# verify shapes and values
print(f"PyTorch attention shape: {torch_atten.shape}")
print(f"JAX attention shape: {jax_atten.shape}")
print(f"Shapes match: {torch_atten.shape == jax_atten.shape}")
print(f"Values close: {np.allclose(torch_atten, jax_atten, rtol=1e-5)}")    
diff = np.array(jax_atten) - activations[layer_name].detach().cpu().numpy()
print(f"Max difference: {np.abs(diff).max()}, Mean difference: {np.abs(diff).mean()}, Std difference: {np.abs(diff).std()}")


Inside Qwen2Attention forward
Removing hook
(1, 10, 151936)
PyTorch attention shape: (1, 10, 896)
JAX attention shape: (1, 10, 896)
Shapes match: True
Values close: False
Max difference: 0.0007171630859375, Mean difference: 6.777048110961914e-05, Std difference: 0.0


In [310]:
def compare_forward_passes(pytorch_model, jax_model, jax_params, test_input_ids, atol=1e-4):
    """
    Compare forward passes between PyTorch and JAX models.
    
    Args:
        pytorch_model: PyTorch model
        jax_model: JAX model
        jax_params: JAX parameters
        test_input_ids: Input token IDs for testing
        atol: Absolute tolerance for comparison
        
    Returns:
        dict: Comparison results
    """
    print("🔍 FORWARD PASS COMPARISON")
    print("=" * 50)
    
    # Prepare inputs
    batch_size, seq_len = test_input_ids.shape
    print(f"Test input shape: {test_input_ids.shape}")
    
    # PyTorch forward pass
    print("\n1️⃣ PyTorch Forward Pass")
    pytorch_input = torch.from_numpy(np.array(test_input_ids)).to(next(pytorch_model.parameters()).device)
    
    with torch.no_grad():
        pytorch_model.eval()
        pytorch_logits = pytorch_model(pytorch_input).logits
    
    print(f"   Input shape: {pytorch_input.shape}")
    print(f"   Output shape: {pytorch_logits.shape}")
    print(f"   Output dtype: {pytorch_logits.dtype}")
    print(f"   Output range: [{pytorch_logits.min().item():.4f}, {pytorch_logits.max().item():.4f}]")
    
    # JAX forward pass
    print("\n2️⃣ JAX Forward Pass")
    jax_input = jnp.array(test_input_ids)
    jax_logits = jax_model.apply(jax_params, jax_input, train=False)
    
    print(f"   Input shape: {jax_input.shape}")
    print(f"   Output shape: {jax_logits.shape}")
    print(f"   Output dtype: {jax_logits.dtype}")
    print(f"   Output range: [{jax_logits.min():.4f}, {jax_logits.max():.4f}]")
    
    # Compare outputs
    print("\n3️⃣ Comparison Results")
    pytorch_logits_np = pytorch_logits.cpu().numpy()
    jax_logits_np = np.array(jax_logits)
    
    # Shape comparison
    shapes_match = pytorch_logits_np.shape == jax_logits_np.shape
    print(f"   Shape match: {'✅' if shapes_match else '❌'}")
    if not shapes_match:
        print(f"      PyTorch: {pytorch_logits_np.shape}")
        print(f"      JAX: {jax_logits_np.shape}")
        return {"success": False, "error": "Shape mismatch"}
    
    # Numerical comparison
    max_diff = np.abs(pytorch_logits_np - jax_logits_np).max()
    mean_diff = np.abs(pytorch_logits_np - jax_logits_np).mean()
    close_match = np.allclose(pytorch_logits_np, jax_logits_np, atol=atol)
    
    print(f"   Max absolute difference: {max_diff:.6f}")
    print(f"   Mean absolute difference: {mean_diff:.6f}")
    print(f"   Close match (atol={atol}): {'✅' if close_match else '❌'}")
    
    # Detailed statistics
    relative_error = np.abs(pytorch_logits_np - jax_logits_np) / (np.abs(pytorch_logits_np) + 1e-8)
    print(f"   Max relative error: {relative_error.max():.6f}")
    print(f"   Mean relative error: {relative_error.mean():.6f}")
    
    # Check specific positions
    print("\n4️⃣ Sample Output Comparison (first 3 positions, first 5 tokens)")
    for pos in range(min(3, seq_len)):
        print(f"   Position {pos}:")
        for token in range(min(5, pytorch_logits_np.shape[-1])):
            pt_val = pytorch_logits_np[0, pos, token]
            jax_val = jax_logits_np[0, pos, token]
            diff = abs(pt_val - jax_val)
            print(f"      Token {token}: PT={pt_val:.4f}, JAX={jax_val:.4f}, diff={diff:.6f}")
    
    return {
        "success": True,
        "shapes_match": shapes_match,
        "close_match": close_match,
        "max_diff": max_diff,
        "mean_diff": mean_diff,
        "max_relative_error": relative_error.max(),
        "pytorch_logits": pytorch_logits_np,
        "jax_logits": jax_logits_np
    }

# Create test input
test_seq_len = 8
test_input_ids = jnp.ones((1, test_seq_len), dtype=jnp.int32) * 50256  # Use a valid token ID

print("Preparing to compare forward passes...")
print(f"Test input: {test_input_ids}")
print(f"Token ID used: {test_input_ids[0, 0]} (should be valid for vocab size {VOCAB_SIZE})")

Preparing to compare forward passes...
Test input: [[50256 50256 50256 50256 50256 50256 50256 50256]]
Token ID used: 50256 (should be valid for vocab size 151936)


In [311]:
# Run the forward pass comparison
try:
    comparison_results = compare_forward_passes(
        pytorch_model=model,
        jax_model=jax_model, 
        jax_params=jax_params,
        test_input_ids=test_input_ids,
        atol=1e-2  # Slightly relaxed tolerance
    )
    
    if comparison_results["success"]:
        print(f"\n🎉 COMPARISON SUMMARY:")
        print(f"   Forward passes {'✅ MATCH' if comparison_results['close_match'] else '❌ DIFFER'}")
        print(f"   Max difference: {comparison_results['max_diff']:.6f}")
        print(f"   Mean difference: {comparison_results['mean_diff']:.6f}")
        
        if comparison_results["close_match"]:
            print(f"\n🎯 SUCCESS: PyTorch and JAX models produce equivalent outputs!")
        else:
            print(f"\n⚠️  Models differ beyond tolerance. Check parameter transfer.")
    
except Exception as e:
    print(f"❌ Error during comparison: {e}")
    import traceback
    traceback.print_exc()

🔍 FORWARD PASS COMPARISON
Test input shape: (1, 8)

1️⃣ PyTorch Forward Pass
   Input shape: torch.Size([1, 8])
   Output shape: torch.Size([1, 8, 151936])
   Output dtype: torch.float16
   Output range: [-8.8281, 11.9766]

2️⃣ JAX Forward Pass
   Input shape: (1, 8)
   Output shape: (1, 8, 151936)
   Output dtype: float16
   Output range: [-8.8750, 12.0000]

3️⃣ Comparison Results
   Shape match: ✅
   Max absolute difference: 0.175781
   Mean absolute difference: 0.016861
   Close match (atol=0.01): ❌
   Max relative error: 2804.000000
   Mean relative error: 0.076965

4️⃣ Sample Output Comparison (first 3 positions, first 5 tokens)
   Position 0:
      Token 0: PT=6.0547, JAX=6.0430, diff=0.011719
      Token 1: PT=9.0234, JAX=9.0312, diff=0.007812
      Token 2: PT=6.6289, JAX=6.6328, diff=0.003906
      Token 3: PT=10.5156, JAX=10.5234, diff=0.007812
      Token 4: PT=5.6406, JAX=5.6328, diff=0.007812
   Position 1:
      Token 0: PT=6.0547, JAX=6.0430, diff=0.011719
      Token 1:

## With qwen2 tokenizer

In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''  # Ensure a GPU is available
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
print(torch.__version__)
print("cuDNN version:", torch.backends.cudnn.version())
print(torch.cuda.is_available())
import jax
print("jax version ", jax.__version__)
# print("jax local devices:", jax.local_devices())

2.7.1+cu128
cuDNN version: 90701
False
jax version  0.5.3


In [2]:
import flax
print("Flax version:", flax.__version__)

Flax version: 0.10.6


In [3]:
MODEL_NAME = "Qwen/Qwen2.5-7B"  # Smaller 0.5B model for easier loading

# Device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Target Device: {device}")

# Model loading configuration (0.5B model is much smaller)
model_config = {
    "device_map": "cpu",
    "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
    "trust_remote_code": False,
}

# Load Qwen 2.5 0.5B Model
print("Loading model...")

try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_config)
    model.eval()
    print("Model loaded!")

    # Basic info
    param_count = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {param_count:,}")

except Exception as e:
    print(f"Error: {e}")
    model = None

Target Device: cpu
Loading model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded!
Parameters: 7,615,616,512


In [26]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (rotary_emb):

In [12]:
for n,p in model.named_parameters():
    print(n, p.shape)

model.embed_tokens.weight torch.Size([152064, 3584])
model.layers.0.self_attn.q_proj.weight torch.Size([3584, 3584])
model.layers.0.self_attn.q_proj.bias torch.Size([3584])
model.layers.0.self_attn.k_proj.weight torch.Size([512, 3584])
model.layers.0.self_attn.k_proj.bias torch.Size([512])
model.layers.0.self_attn.v_proj.weight torch.Size([512, 3584])
model.layers.0.self_attn.v_proj.bias torch.Size([512])
model.layers.0.self_attn.o_proj.weight torch.Size([3584, 3584])
model.layers.0.mlp.gate_proj.weight torch.Size([18944, 3584])
model.layers.0.mlp.up_proj.weight torch.Size([18944, 3584])
model.layers.0.mlp.down_proj.weight torch.Size([3584, 18944])
model.layers.0.input_layernorm.weight torch.Size([3584])
model.layers.0.post_attention_layernorm.weight torch.Size([3584])
model.layers.1.self_attn.q_proj.weight torch.Size([3584, 3584])
model.layers.1.self_attn.q_proj.bias torch.Size([3584])
model.layers.1.self_attn.k_proj.weight torch.Size([512, 3584])
model.layers.1.self_attn.k_proj.bias 

In [20]:
model.model.layers[3].self_attn.q_proj.weight

Parameter containing:
tensor([[-0.0057, -0.0001, -0.0081,  ..., -0.0197,  0.0140, -0.0171],
        [ 0.0119,  0.0104, -0.0059,  ..., -0.0177,  0.0400, -0.0210],
        [-0.0078,  0.0288,  0.0084,  ...,  0.0150, -0.0036, -0.0161],
        ...,
        [ 0.0066, -0.0055,  0.0083,  ...,  0.0020, -0.0369, -0.0058],
        [ 0.0008,  0.0150, -0.0029,  ...,  0.0165,  0.0055,  0.0048],
        [ 0.0065, -0.0659,  0.0092,  ..., -0.0286,  0.0017,  0.0226]],
       requires_grad=True)

In [6]:
from src.qwen_jax import convert_pytorch_weights_to_jax, QwenForCausalLM
from save_datasets_and_params import create_qwen_config_from_pytorch
import jax
import jax.numpy as jnp 

# Create model configuration from PyTorch config
jax_config = create_qwen_config_from_pytorch(model.config)
print("JAX model configuration:")
for key, value in jax_config.items():
    print(f"   {key}: {value}")

# Initialize full JAX model
print("\nInitializing QwenForCausalLM...")
jax_model = QwenForCausalLM(**jax_config)

rng = jax.random.PRNGKey(42)
dummy_input_ids = jnp.ones((1, 10), dtype=jnp.int32)
jax_params = jax_model.init(rng, dummy_input_ids)

# Convert and transfer weights from PyTorch to JAX
jax_params = convert_pytorch_weights_to_jax(jax_params, model, jax_config, jax_dtype=jnp.bfloat16)


JAX model configuration:
   vocab_size: 152064
   hidden_size: 3584
   num_hidden_layers: 28
   num_attention_heads: 28
   num_key_value_heads: 4
   intermediate_size: 18944
   hidden_act: silu
   max_position_embeddings: 131072
   rms_norm_eps: 1e-06
   attention_dropout: 0.0
   tie_word_embeddings: False
   rope_theta: 1000000.0

Initializing QwenForCausalLM...


: 

In [ ]:
for n,p in model.named_parameters():
    print(n, p.shape)

In [27]:
id(model.lm_head.weight)

140516568601344

In [28]:
id(model.model.embed_tokens.weight)

140516567307728

In [1]:
model.model.embed_tokens.weight.shape, model.lm_head.weight.shape

NameError: name 'model' is not defined

In [24]:
jax_params['params']['embed_tokens']['embedding']

Array([[-0.0100708, 0.0419922, 0.0098877, ..., 0.00994873, 0.0151367,
        -0.00704956],
       [-0.0146484, 6.03199e-05, -0.0184326, ..., -0.00169373,
        0.00311279, -0.00842285],
       [-0.0366211, -0.00982666, 0.00698853, ..., -0.00775146,
        -0.0170898, -0.000104904],
       ...,
       [0.0027771, -0.00939941, 0.00424194, ..., -0.00656128, -0.0106201,
        0.0172119],
       [0.0027771, -0.00939941, 0.00424194, ..., -0.00656128, -0.0106201,
        0.0172119],
       [0.0027771, -0.00939941, 0.00424194, ..., -0.00656128, -0.0106201,
        0.0172119]], dtype=bfloat16)

In [6]:

from transformers import Qwen2Tokenizer

tokenizer = Qwen2Tokenizer.from_pretrained("Qwen/Qwen2-0.5B")

In [10]:
tokenizer

Qwen2Tokenizer(name_or_path='Qwen/Qwen2-0.5B', vocab_size=151643, model_max_length=32768, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [28]:
@jax.jit
def jit_apply(input_ids):
    return jax_model.apply(jax_params, input_ids, train=False)

In [8]:
input_tokens = tokenizer("Hallo, You are such an excellent poet. Can you tell me a story?", return_tensors="np")
print(f"input token length {input_tokens['input_ids'].shape}")
input = input_tokens["input_ids"]

# torch
t_out = model(torch.from_numpy(input).to(next(model.parameters()).device))
print(tokenizer.decode(t_out.logits[0].argmax(dim=-1).cpu().numpy()))

# jax
j_out = jax_model.apply(jax_params, input, train=False)
print(tokenizer.decode(torch.from_numpy(np.array(j_out[0].argmax(axis=-1)))))


input token length (1, 16)
, ich are looking a awesome writer. I you please me more little about I
, ich are looking a awesome writer. I you please me more little about I


In [22]:
%%timeit

t_out = model(torch.from_numpy(input).to(next(model.parameters()).device))

21.6 ms ± 921 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [23]:
%%timeit
j_out = jax_model.apply(jax_params, input, train=False)

533 ms ± 2.22 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [27]:
# generate text with jax model

def jax_generate_text(prompt, max_new_tokens=10):
    inputs = tokenizer(prompt, return_tensors="np")["input_ids"]
    print(f"Prompt: '{prompt}'")
    print(f"Input Tokens: {inputs}")
    
    for _ in range(max_new_tokens):
        logits = jax_model.apply(jax_params, inputs, train=False)
        next_token = jnp.argmax(logits[0, -1, :])
        inputs = jnp.concatenate([inputs, next_token[None, None]], axis=-1)
    
    generated_text = tokenizer.decode(np.array(inputs[0]), skip_special_tokens=True)
    print(f"Generated: '{generated_text}'")
    return generated_text

# Test the function
jax_generate_text("How can I transform a pytorch model into jax?", max_new_tokens=20)



Prompt: 'How can I transform a pytorch model into jax?'
Input Tokens: [[ 4340   646   358  5165   264  4510 27414  1614  1119   502   706    30]]
Next token: 358 () ( I)
Updated input shape: (1, 13)
Next token: 614 () ( have)
Updated input shape: (1, 14)
Next token: 264 () ( a)
Updated input shape: (1, 15)
Next token: 4510 () ( py)
Updated input shape: (1, 16)
Next token: 27414 () (torch)
Updated input shape: (1, 17)


2025-08-28 11:59:41.896682: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-08-28 11:59:43.714638: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


Next token: 1614 () ( model)
Updated input shape: (1, 18)


2025-08-28 11:59:47.558300: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-08-28 11:59:49.164222: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


Next token: 429 () ( that)
Updated input shape: (1, 19)


2025-08-28 11:59:52.978267: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-08-28 11:59:54.616695: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


... (20 outputs left out)

Next token: 614 () ( have)
Updated input shape: (1, 31)


2025-08-28 12:00:53.831832: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-08-28 12:00:55.561375: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


Next token: 6679 () ( tried)
Updated input shape: (1, 32)
Generated: 'How can I transform a pytorch model into jax? I have a pytorch model that I want to convert into a jax model. I have tried'


'How can I transform a pytorch model into jax? I have a pytorch model that I want to convert into a jax model. I have tried'

In [20]:
tokenizer("Hello, how are you?", return_tensors="pt").to('cuda')

{'input_ids': tensor([[9707,   11, 1246,  525,  498,   30]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [334]:
t_out = model.generate(**tokenizer("Hello, can you tell me a story?", return_tensors="pt").to('cuda'), max_new_tokens=100, do_sample=False)
print(tokenizer.decode(t_out[0].cpu().numpy()))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Hello, can you tell me a story? I want to write a story about a girl who is a superhero. Sure, I'd be happy to help you with that! Here's a story about a girl who becomes a superhero:

Once upon a time, in a small town, there was a young girl named Lily. She was always curious and adventurous, and she loved exploring the world around her. One day, while playing in the park, she stumbled upon a mysterious object that seemed to be made of metal. It was a strange,
